# Hot/Cold Blank Filling + Average SAE Activations

This notebook mirrors the affectations2 workflow using a trainable SAE.

In [18]:
from pathlib import Path
import random
import sys

import pandas as pd
import torch

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.append(Path.home() / 'Projects/MATH498-Sparse-Autoencoder-Manipulation')
    for candidate in candidates:
        if (candidate / 'trainable_sae.py').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing trainable_sae.py')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from trainable_sae import SAEConfig, TrainableSAE, SAEConnector, HookPointSpec, load_hooked_transformer

## Build hot/cold sentence pairs

In [19]:
HOT_WORDS  = ['hot']#],'warm', 'heated']#'scorching','blazing','sweltering','sizzling','boiling','torrid','searing','broiling']
COLD_WORDS = ['cold']#, 'chilly','frigid']#'freezing','frigid','frosty','glacial','wintry','icy','nippy','arctic']

fib_path = PROJECT_ROOT / 'experiment_scripts/experiment_data/affectations2/fib_sentences.txt'
lines = [line.strip() for line in fib_path.read_text(encoding='utf-8').splitlines() if line.strip()]

hot_sentences = []
cold_sentences = []
for line in lines:
    body = ' '.join(line.split()[1:])
    blank_idx = body.index('<blank>')
    prefix, suffix = body[:blank_idx], body[blank_idx + len('<blank>'):]
    for hot_word in HOT_WORDS:
        hot_sentence = prefix + hot_word + suffix
        hot_sentences.append(hot_sentence)
    for cold_word in COLD_WORDS:
        cold_sentence = prefix + cold_word + suffix
        cold_sentences.append(cold_sentence)

hot_sentences[:3], cold_sentences[:3]

(['The demeanor was hot enough.',
  'The bathroom felt hot.',
  'The tap water became hot.'],
 ['The demeanor was cold enough.',
  'The bathroom felt cold.',
  'The tap water became cold.'])

## Load model + trained SAE

In [20]:
MODEL_NAME = 'google/gemma-3-270m-it'
SAE_PATH = PROJECT_ROOT / 'custom_saes/relu_mid_1/relu'
SAE_PATH = PROJECT_ROOT / 'saved_saes/shrink_mid_1/shrink/best_step_update'

device = 'cuda:2' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = load_hooked_transformer(MODEL_NAME, device=device, dtype=dtype)
model.to(dtype)
sae = TrainableSAE.load(SAE_PATH, device=device)
sae.eval()

hook_point = sae.cfg.metadata.get(
    'hook_point',
    HookPointSpec(layer=model.cfg.n_layers // 2, site='resid_post').name,
 )

connector = SAEConnector(
    model=model,
    sae=sae,
    hook_point=hook_point,
    device=device,
    preserve_error=True,
 )

print(f'model:      {MODEL_NAME}')
print(f'sae path:   {SAE_PATH}')
print(f'hook point: {hook_point}')
print(f'sae:        {sae.d_in} -> {sae.d_sae}, activation={sae.cfg.activation}')

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 15038.83it/s]


Loaded pretrained model google/gemma-3-270m-it into HookedTransformer
Moving model to device:  cuda:2
Changing model dtype to torch.bfloat16
Changing model dtype to torch.bfloat16
Moving model to device:  cuda:2
model:      google/gemma-3-270m-it
sae path:   /home/andyholmberg/Projects/MATH498-Sparse-Autoencoder-Manipulation/saved_saes/shrink_mid_1/shrink/best_step_update
hook point: blocks.9.hook_resid_post
sae:        640 -> 81920, activation=shrink


## Compute average activations (hot vs cold)

In [21]:
def encode_sentence(sentence: str) -> torch.Tensor:
    tokens = model.to_tokens(sentence).to(device)
    with torch.no_grad():
        acts = connector.collect_activations(tokens)
        feats = sae.encode(acts)
    return feats.reshape(-1, sae.d_sae).to(dtype=torch.float32)

hot_sums = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)
cold_sums = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)
hot_counts = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)
cold_counts = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)

for hot_sentence in hot_sentences:
    hot_feats = encode_sentence(hot_sentence)
    hot_sums += hot_feats.mean(dim=0)
    hot_counts += 1

for cold_sentence in cold_sentences:
    cold_feats = encode_sentence(cold_sentence)
    cold_sums += cold_feats.mean(dim=0)
    cold_counts += 1

In [22]:

avg_hot = (hot_sums / hot_counts).detach().cpu()
avg_cold = (cold_sums / cold_counts).detach().cpu()
delta_avg = avg_hot - avg_cold

top_k_values, _ = torch.topk(torch.abs(delta_avg), k=1000)

# 2. Get the minimum value among the top k
min_top_k = top_k_values[-1]

# 3. Create a mask and zero out other values
# Everything strictly less than the k-th value becomes 0
delta_avg[torch.abs(delta_avg) < min_top_k] = 0

df = pd.DataFrame({
    'feature_id': range(sae.d_sae),
    'avg_hot': avg_hot.numpy(),
    'avg_cold': avg_cold.numpy(),
    'delta_avg': delta_avg.numpy(),
})
df['abs_delta'] = df['delta_avg'].abs()
df.sort_values('abs_delta', ascending=False).head(10)

,feature_id,avg_hot,avg_cold,delta_avg,abs_delta
45011,45011,10.463205,9.063957,1.399248,1.399248
37262,37262,3.109884,1.823960,1.285924,1.285924
18127,18127,6.832086,5.558179,1.273907,1.273907
59609,59609,8.358092,7.235579,1.122513,1.122513
46131,46131,4.741611,3.725869,1.015742,1.015742
30631,30631,-6.994234,-7.992943,0.998709,0.998709
63671,63671,-7.551489,-6.572820,-0.978669,0.978669
62079,62079,-4.910919,-3.984773,-0.926146,0.926146
39451,39451,-3.635186,-2.776436,-0.858750,0.858750
58869,58869,-3.572177,-4.420047,0.847870,0.847870


## Generate with the hot feature vector

In [23]:

hot_vector = delta_avg.to(device=device, dtype=next(sae.parameters()).dtype)


def format_prompt_for_model(prompt: str) -> str:
    model_name = str(getattr(model.cfg, "model_name", "") or "").lower()
    if "gemma" in model_name and "it" in model_name:
        return f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    return prompt


def reset_generation_seed(seed: int | None) -> None:
    if seed is None:
        return
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_modified_responses_add(
    gen_prompt,
    max_new_tokens,
    hot_strength,
    temperature,
    seed=0,
    sae_enabled=True,
    do_sample=False,
    top_p=0.95,
    token_index='all',
    projector_location="post_activation",
):
    def add_hot_vector(features: torch.Tensor) -> torch.Tensor:    
        return features + hot_strength * hot_vector

    def add_cold_vector(features: torch.Tensor) -> torch.Tensor:
        return features - hot_strength * hot_vector

    prompt = format_prompt_for_model(gen_prompt)
    generate_kwargs = dict(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        use_past_kv_cache=False,
        verbose=False,
    )

    tokens = model.to_tokens(prompt).to(device)
    prompt_len = tokens.shape[1]

    reset_generation_seed(seed)
    with torch.no_grad():
        baseline_tokens = model.generate(tokens, **generate_kwargs)
    baseline = model.to_string(baseline_tokens[0, prompt_len:]).strip()

    reset_generation_seed(seed)
    if not sae_enabled:
        hot_steered = baseline
        cold_steered = baseline
    else:
        with torch.no_grad():
            hot_steered = connector.generate_with_sae(
                prompt,
                mode="reconstruct",
                sae_projector=add_hot_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )

        with torch.no_grad():
            cold_steered = connector.generate_with_sae(
                prompt,
                mode="reconstruct",
                sae_projector=add_cold_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )
    print("--- baseline ---")
    print(baseline)
    print("\n--- + hot feature vector ---")
    print(hot_steered)

    print("\n--- - cold feature vector ---")
    print(cold_steered)

GEN_PROMPT = "is the temperature outside hot or cold?"
MAX_NEW_TOKENS = 100
HOT_STRENGTH = 10


print_modified_responses_add(GEN_PROMPT, MAX_NEW_TOKENS, HOT_STRENGTH, .8, seed=42)
# df.sort_values('delta_avg', ascending=False).head(10)

--- baseline ---
The temperature outside is generally considered to be **hot**.
<end_of_turn><end_of_turn><end_of_turn>
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>

<end_of_turn><end_of_turn><end_of_turn><end_of_turn>

<end_of_turn><end_of_turn><end_of_turn><end_of_turn>

<end_of_turn><end_of_turn><end_of_turn><end_of_turn>
<e

In [24]:
GEN_PROMPT = "give me a brief description of the weather outside"
MAX_NEW_TOKENS = 40
HOT_STRENGTH = 10

print_modified_responses_add(GEN_PROMPT, MAX_NEW_TOKENS, HOT_STRENGTH, .8, seed=42)


--- baseline ---
The weather outside is typically sunny with a high of around 70 degrees Fahrenheit and a low of around 50 degrees Fahrenheit.
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>

--- + hot feature vector ---
The weather outside is typically sunny and warm. It can range from sunny and bright to cool and breezy.<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>

--- - cold feature vector ---
The weather outside is typically sunny with a high of around 25°C (77°F) and a low of around 10°C (50°F).
<end_of_turn>


In [25]:
GEN_PROMPT = "do I need a jacket for today?"
MAX_NEW_TOKENS = 40
HOT_STRENGTH = 15

print_modified_responses_add(GEN_PROMPT, MAX_NEW_TOKENS, HOT_STRENGTH, 0, seed=42)


--- baseline ---
Yes, you definitely need a jacket for today!
<end_of_turn><end_of_turn><end_of_turn>
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>

--- + hot feature vector ---
No, you don't need a jacket for today.
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>

--- - cold feature vector ---
Yes, you absolutely need a jacket for today.
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn><end_of_turn><end_of

In [26]:
GEN_PROMPT = "tell a story about a hot day"
MAX_NEW_TOKENS = 30
HOT_STRENGTH = 20

print_modified_responses_add(GEN_PROMPT, MAX_NEW_TOKENS, HOT_STRENGTH, .8, seed=42)


--- baseline ---
The sun beat down on the dusty plains of Dustbowl, baking the earth to a scorching inferno. A relentless heatwave had gripped the region,

--- + hot feature vector ---
The air crackled with the scent of burnt sugar and impending doom. It was a hot day, a day that felt like a furnace in the summer

--- - cold feature vector ---
The sun beat down on the parched earth, baking the parched land into a fiery orange. A relentless heat, a relentless thirst, was the


In [27]:
GEN_PROMPT = "do I need a blanket today?"
MAX_NEW_TOKENS = 30
HOT_STRENGTH = 10

print_modified_responses_add(GEN_PROMPT, MAX_NEW_TOKENS, HOT_STRENGTH, .8, seed=42)


--- baseline ---
Yes, you definitely need a blanket today! <end_of_turn>
<end_of_turn>
<end_of_turn><end_of_turn><end_of_turn><end_of_turn>
<end_of_turn><end_of_turn><end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>

--- + hot feature vector ---
No, you don't need a blanket today. 
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>

--- - cold feature vector ---
Yes, you absolutely need a blanket today.
<end_of_turn>
<end_of_turn>
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>


In [28]:
hot_vector = delta_avg.to(device=device, dtype=next(sae.parameters()).dtype)
delta_avg = delta_avg.to(device=device, dtype=next(sae.parameters()).dtype)
def format_prompt_for_model(prompt: str) -> str:
    model_name = str(getattr(model.cfg, 'model_name', '') or '').lower()
    if 'gemma' in model_name and 'it' in model_name:
        return f'<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n'
    return prompt

def reset_generation_seed(seed: int | None) -> None:
    if seed is None:
        return
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def print_modified_responses_project(
    gen_prompt,
    max_new_tokens,
    correction_alpha,
    temperature,
    seed=0,
    sae_enabled=True,
    do_sample=False,
    top_p=0.95,
    token_index='all',
    projector_location='post_activation',
):
    def add_hot_vector(features: torch.Tensor) -> torch.Tensor:
        b = 0
        a = hot_vector
        if features.T@a > b:
            return features
        else:
            return features - correction_alpha * (features.T@a - b) / (a.T@a) * a
    def add_cold_vector(features: torch.Tensor) -> torch.Tensor:
        b = 0
        a = -hot_vector
        if features.T@a > b:
            # print("SAD")
            return features
        else:
            # print("Too hot - correcting...")
            return features - correction_alpha * (features.T@a - b) / (a.T@a) * a
        
    prompt = format_prompt_for_model(gen_prompt)
    generate_kwargs = dict(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        use_past_kv_cache=False,
        verbose=False,
    )

    tokens = model.to_tokens(prompt).to(device)
    prompt_len = tokens.shape[1]

    reset_generation_seed(seed)
    with torch.no_grad():
        baseline_tokens = model.generate(tokens, **generate_kwargs)
    baseline = model.to_string(baseline_tokens[0, prompt_len:]).strip()

    reset_generation_seed(seed)
    if not sae_enabled:
        hot_steered = baseline
        cold_steered = baseline
    else:
        with torch.no_grad():
            hot_steered = connector.generate_with_sae(
                prompt,
                mode='reconstruct',
                sae_projector=add_hot_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )

        with torch.no_grad():
            cold_steered = connector.generate_with_sae(
                prompt,
                mode='reconstruct',
                sae_projector=add_cold_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )
    print('--- baseline ---')
    print(baseline)
    print('\n--- + hot feature vector ---')
    print(hot_steered)
    print('\n--- - cold feature vector ---')
    print(cold_steered)

In [29]:
GEN_PROMPT = "tell me a story about the cold wind"
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 0

print_modified_responses_project(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
The wind howled a mournful song, a symphony of cold and white that whipped across the desolate landscape. It wasn't a gentle breeze, but a furious, biting gale that threatened to tear apart the fragile beauty of the valley. The valley was a tapestry of green, a vibrant green that

--- + hot feature vector ---
The wind howled a mournful song, a symphony of cold and white that whipped across the desolate landscape. It wasn't a gentle breeze, but a furious, biting gale that threatened to tear apart the fragile beauty of the valley. The valley was a tapestry of green, a vibrant green that

--- - cold feature vector ---
The wind howled a mournful song, a symphony of cold and white that whipped across the desolate landscape. It wasn't a gentle breeze, but a furious, biting gale that threatened to tear apart the fragile beauty of the valley. The valley was a tapestry of green, a vibrant green that


In [30]:
GEN_PROMPT = "tell a story about mt everest"
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 1

print_modified_responses_project(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
The wind whipped at Anya's face, a furious dance of wind and cold. She stood on the precipice, the world shrinking below her, a dizzying expanse of green and blue. Below, the jagged peaks of the Himalayas clawed at the sky, their snow-capped summits piercing the

--- + hot feature vector ---
The wind whipped at Anya's face, a furious dance of white against the azure sky. She gripped the railing of the summit, her knuckles white, her heart a frantic drum against her ribs. The world stretched out below, a tapestry of emerald green and sapphire blue, a breathtaking panorama of the

--- - cold feature vector ---
The wind whipped at Anya's face, a furious dance of wind and cold. She stood on the precipice, the world shrinking below her, a dizzying expanse of green and blue. Below, the jagged peaks of the Himalayas clawed at the sky, their snow-capped summits a stark


In [31]:
GEN_PROMPT = "tell me a story about a snowman"
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 1

print_modified_responses_project(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
In a world of frosted smiles and cheerful greetings, lived a snowman named Olaf. He wasn't just any snowman; he was a creature of pure, unadulterated joy. His coat was a vibrant, cheerful yellow, his eyes sparkled with mischief, and his laughter was a tinkling

--- + hot feature vector ---
The wind whipped through the skeletal branches of the old oak tree, rustling the leaves in a symphony of whispers. Snowdrift was a solitary figure, a pale, almost translucent silhouette against the grey sky. He stood tall, a monument to winter's cold embrace, his face a roadmap of wrinkles

--- - cold feature vector ---
The wind whipped through the skeletal branches of the snowman's frame, carrying the scent of pine and damp earth. He stood tall, a majestic figure sculpted from snow and frost, his face a mask of silent determination. His eyes, the color of a summer sky, held a deep, knowing gaze


In [32]:
GEN_PROMPT = "will I need a blanket today?"
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 2

print_modified_responses_project(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
No, you will not need a blanket today.
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn>

--- + hot feature vector ---
No, you will not need a blanket today.
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><en

In [33]:
GEN_PROMPT = "tell me a story about a princess"
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 2

print_modified_responses_project(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
Princess Aurelia, with her silver hair braided with wildflowers and eyes the color of a summer sky, was known throughout the kingdom of Eldoria for her kindness and her unwavering spirit. She was a princess, yes, but more than that, she was a beacon of hope, a queen who understood the

--- + hot feature vector ---
Princess Aurelia, with her silver hair braided with wildflowers and eyes the color of a summer sky, was known throughout the kingdom of Eldoria for her kindness and her unwavering spirit. She was a princess, yes, but more than that, she was a beacon of hope, a queen who understood the

--- - cold feature vector ---
Princess Aurelia, with her silver hair braided with wildflowers and eyes the color of the summer sky, was known throughout the kingdom of Eldoria for her kindness and her unwavering spirit. She was a princess of the Silverwood, a place where the ancient trees whispered secrets to the wind and the rivers flowed
